In [3]:
# =============================================================================
# QUANTIZATION & INFERENCE OPTIMIZATION — make Correction-GPT faster/smaller
# =============================================================================
#
# ---------------------------------------------------------------------------
# WHERE ARE WE ON THE PATH?
# ---------------------------------------------------------------------------
#   Pretrain → SFT → DPO  = teach the model WHAT to say
#   THIS NOTEBOOK         = serve it CHEAPER / FASTER on a device
#
# Easy analogy:
#   Training notebooks = write the book.
#   Quantization       = print a pocket edition (smaller file, still readable).
#   Inference opt      = read it faster (less waiting for the next word).
#
# Why care for your earpiece product?
#   Live "Correction: ..." whispers need LOW latency on phone/edge hardware.
#   Full float32 weights are accurate but heavy. Quantization shrinks them.
#
#
# ---------------------------------------------------------------------------
# THIS CELL — load the trained model + measure BASELINE speed/size
# ---------------------------------------------------------------------------
# Before optimizing, measure the "before" numbers:
#   1) How long to generate ~15 tokens?  (latency in ms)
#   2) How many parameters?
#   3) How many MB if each weight is float32 (4 bytes)?
# Later cells compare quantized / optimized versions to THIS baseline.
#

from pathlib import Path
import sys
import time

import torch

# week2/ on path — same pattern as notebooks 9–10
# (You cannot `import` a .ipynb; shared code lives in .py modules.)
_WEEK2 = Path.cwd() / "week2"
if _WEEK2.is_dir():
    sys.path.insert(0, str(_WEEK2.resolve()))
elif Path.cwd().name == "week2":
    sys.path.insert(0, str(Path.cwd().resolve()))
else:
    sys.path.insert(0, str(Path("week2").resolve()))

# MiniGPT = model class from notebook 7, saved as week2/mini_gpt.py
#   NOT day8_minigpt — that module name does not exist.
from mini_gpt import MiniGPT

# BPETokenizer = from notebook 8, saved as week2/tokenizer.py
#   Sticky lesson from notebook 8: keep </w> inside merges (don't strip)
#   or decode glues words like "theapirate...".
import importlib
import tokenizer as _tokenizer_mod
importlib.reload(_tokenizer_mod)  # pick up save/load if kernel had an old copy
from tokenizer import BPETokenizer

# ---------------------------------------------------------------------------
# Load the SAME tokenizer the DPO/SFT weights were trained with
# ---------------------------------------------------------------------------
# Do NOT train on "placeholder" text — that makes a NEW vocab (wrong size)
# and torch.load will crash with shape mismatch (260 vs something else).
#
# Picture:
#   correction_gpt_sft_tokenizer.json  →  jersey numbers (token → id)
#   correction_gpt_dpo.pt              →  weights that EXPECT those ids
#

_tok_candidates = [
    Path("correction_gpt_sft_tokenizer.json"),
    Path("week2") / "correction_gpt_sft_tokenizer.json",
]
_tok_path = next((p for p in _tok_candidates if p.is_file()), None)
if _tok_path is None:
    raise FileNotFoundError(
        "Need correction_gpt_sft_tokenizer.json (from notebook 9). "
        "Run SFT save cell or keep the file in week2/."
    )

tokenizer = BPETokenizer()
tokenizer.load(_tok_path)

vocab_size = len(tokenizer.vocab)
block_size = 64  # must match checkpoint pos_embed length

model = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)

# Prefer DPO weights; fall back to SFT if DPO file missing
_ckpt_candidates = [
    Path("correction_gpt_dpo.pt"),
    Path("week2") / "correction_gpt_dpo.pt",
    Path("correction_gpt_sft.pt"),
    Path("week2") / "correction_gpt_sft.pt",
]
_ckpt = next((p for p in _ckpt_candidates if p.is_file()), None)
if _ckpt is None:
    raise FileNotFoundError("Need correction_gpt_dpo.pt or correction_gpt_sft.pt in week2/")

model.load_state_dict(torch.load(_ckpt, map_location="cpu", weights_only=True))
model.eval()  # inference mode: no dropout training behavior
print(f"loaded weights: {_ckpt}")
print(f"tokenizer: {_tok_path} | vocab={vocab_size} | block_size={block_size}")

# ---------------------------------------------------------------------------
# Baseline: time one short generation
# ---------------------------------------------------------------------------
# prompt → encode to ids → generate 15 new tokens → decode
# We only print NEW tokens (not the whole prompt again).
#
prompt = "Ground truth: Price is $500/month\nUser said: $300\nCorrection:"
prompt_ids = tokenizer.encode(prompt)
input_ids = torch.tensor([prompt_ids], dtype=torch.long)

start = time.time()
with torch.no_grad():  # no gradients → faster / less memory at inference
    out = model.generate(input_ids, max_new_tokens=15, temperature=0.7)
end = time.time()

new_ids = out[0, len(prompt_ids) :].tolist()
print(f"Output (new tokens): {tokenizer.decode(new_ids)}")
print(f"Baseline latency: {(end - start) * 1000:.2f} ms")

n_params = sum(p.numel() for p in model.parameters())
bytes_f32 = n_params * 4  # float32 = 4 bytes per number
print(f"Model size (params): {n_params:,}")
print(f"Model size (MB): {bytes_f32 / 1e6:.2f} MB (float32)")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# loaded weights: .../correction_gpt_dpo.pt
#   Using the preference-tuned checkpoint when available.
#
# tokenizer ... vocab=260 | block_size=64
#   MUST match the checkpoint (embed is 260×64, positions length 64).
#
# Output (new tokens): ...
#   Toy model text may still look rough — this cell cares about SPEED/SIZE,
#   not perfect English. Later cells shrink/speed this same generate() call.
#
# Baseline latency: e.g. 5–50 ms (CPU varies a lot)
#   Your "before" number. Quantization should aim lower or similar with
#   smaller memory.
#
# Model size (params): ~137k
# Model size (MB): ~0.55 MB (float32)
#   Sticky: MB ≈ params × 4 / 1e6 for float32.
#   Int8 quantization idea later: ~params × 1 byte → roughly 4× smaller.


Loaded tokenizer ← correction_gpt_sft_tokenizer.json (vocab=260)
loaded weights: correction_gpt_dpo.pt
tokenizer: correction_gpt_sft_tokenizer.json | vocab=260 | block_size=64
Output (new tokens): : rrest respona contraprodpricpt . businnappropriatr
Baseline latency: 15.19 ms
Model size (params): 137,604
Model size (MB): 0.55 MB (float32)


In [ ]:
# =============================================================================
# QUANTIZE TO INT8 — shrink Linear weights (~4× for those layers)
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: what is quantization? (simple English)
# ---------------------------------------------------------------------------
# Training stored each knob as float32 (4 bytes) — like writing money with
# many decimal places.
#
# Quantization ≈ round knobs to fewer bits (here INT8 = 1 byte).
#   Smaller file, often faster matmuls on CPU, tiny quality loss.
#
# Picture:
#   float32 weight:  0.123456789   (4 bytes)
#   int8 version:    scale + small integer -128..127  (≈1 byte + shared scale)
#
#
# ---------------------------------------------------------------------------
# WHY THE OLD STATIC PTQ CELL CRASHED
# ---------------------------------------------------------------------------
# Old code used:
#   model.qconfig = get_default_qconfig('fbgemm')
#   prepare → calibrate → convert
#
# Error you saw:
#   Embedding quantization only works with float_qparams_weight_only_qconfig
#
# Also on this Mac, engine is usually 'qnnpack' (not 'fbgemm'), and full
# static PTQ struggles with our custom MiniGPT (LayerNorm + GELU + hand-rolled
# attention). So we use a friendlier lesson path:
#
#   torchao: Int8DynamicActivationInt8WeightConfig on Linear matmuls
#   - Embeddings stay float
#   - Linear layers (attention/MLP/head) → INT8 weights (+ dynamic INT8 acts)
#
# Sticky: we are NOT quantizing "the whole brain" — mainly the big Linear knobs.
#

import copy
import os
import time

import torch
import torch.nn as nn

# ---------------------------------------------------------------------------
# API note (the DeprecationWarning you saw)
# ---------------------------------------------------------------------------
# torch.ao.quantization (quantize_dynamic, prepare/convert) is DEPRECATED and
# planned for removal in PyTorch 2.10.
# New home: torchao  →  quantize_(model, Int8DynamicActivationInt8WeightConfig())
#
# Same idea as before: INT8 on Linear-style matmuls; embeddings stay float.
# Install once:  pip install torchao
#

from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig

# Fresh FP32 copy for fair timing compare
model_fp32 = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)
model_fp32.load_state_dict(model.state_dict())
model_fp32.eval()

# INT8 dynamic activation + INT8 weights on supported Linear layers (torchao)
model_int8 = MiniGPT(
    vocab_size,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=3,
    block_size=block_size,
)
model_int8.load_state_dict(model.state_dict())
model_int8.eval()
quantize_(model_int8, Int8DynamicActivationInt8WeightConfig())  # in-place
# head.weight becomes an Int8Tensor subclass — that IS the quantization.

# ---- latency: same prompt / same 15 new tokens as baseline cell ----
prompt_ids = tokenizer.encode(prompt)
input_ids = torch.tensor([prompt_ids], dtype=torch.long)

# FP32 timing
start = time.time()
with torch.no_grad():
    out_fp32 = model_fp32.generate(input_ids, max_new_tokens=15, temperature=0.7)
t_fp32 = (time.time() - start) * 1000

# INT8 dynamic timing
start = time.time()
with torch.no_grad():
    out_int8 = model_int8.generate(input_ids, max_new_tokens=15, temperature=0.7)
t_int8 = (time.time() - start) * 1000

print("FP32 output (new):", tokenizer.decode(out_fp32[0, len(prompt_ids) :].tolist()))
print("INT8 output (new):", tokenizer.decode(out_int8[0, len(prompt_ids) :].tolist()))
print(f"FP32 latency: {t_fp32:.2f} ms")
print(f"INT8 latency: {t_int8:.2f} ms")

# ---- size: save state dicts and compare file bytes ----
torch.save(model_fp32.state_dict(), "model_fp32.pth")
torch.save(model_int8.state_dict(), "model_int8.pth")
fp32_kb = os.path.getsize("model_fp32.pth") / 1e3
int8_kb = os.path.getsize("model_int8.pth") / 1e3
print(f"FP32 file: {fp32_kb:.2f} KB")
print(f"INT8 file: {int8_kb:.2f} KB  (Linear weights packed as int8)")

# What got quantized? List Linear modules (these are the INT8 targets)
print("\nLinear layers quantized (dynamic INT8):")
for name, mod in model.named_modules():
    if isinstance(mod, nn.Linear):
        print(f"  - {name}: {tuple(mod.weight.shape)}")

print("\nLeft in float (not dynamic-quantized here):")
print("  - token_embed (nn.Embedding)")
print("  - pos_embed (nn.Parameter)")
print("  - LayerNorm weights")

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT (walkthrough of your run)
# ---------------------------------------------------------------------------
# W... redirects.py: NOTE: Redirects are currently not supported in MacOs.
#   Harmless torchao/distributed startup note on Mac. Ignore for this lesson.
#
# W... _pytree.py: <enum 'KernelPreference'> ... register_constant() deprecated
#   Harmless library warning inside torchao/torch.compile plumbing.
#   Not your bug; does not break generate().
#
# FP32 output / INT8 output (new tokens)
#   Both look like babble on this TOY Correction-GPT — expected.
#   This cell measures SIZE/SPEED, not English quality.
#   Sampling is random (temperature=0.7) so the two strings differ; that does
#   NOT mean INT8 "broke" the model by itself.
#
# FP32 latency: ~10 ms
# INT8 latency: ~34 ms   ← INT8 SLOWER here
#   Sticky: on a TINY model, INT8 packing/unpack + dynamic quant overhead can
#   COST more than it saves. Speedups show up on BIGGER Linears / with
#   torch.compile / real deploy kernels — not guaranteed on MiniGPT@64-dim.
#   So "quantization = always faster" is FALSE for this demo; "smaller" is TRUE.
#
# FP32 file: ~564 KB
# INT8 file: ~237 KB
#   Win: roughly half the bytes on disk because Linear weights are INT8.
#   Not a full 4× shrink because embeddings, pos_embed, LayerNorm, and file
#   format overhead stay in the save.
#
# Linear layers quantized ...
#   blocks.*.attn.qkv / proj  = attention projections
#   blocks.*.ff.0 / ff.2      = MLP expand/shrink
#   head                      = vocab scorer
#   Shapes like (192, 64) = 3*embed for QKV; (260, 64) = vocab × embed.
#   THESE are what we quantized.
#
# Left in float
#   token_embed / pos_embed / LayerNorm — still float32 knobs.

# Note: Real‑world LLMs use GPTQ/AWQ/GGUF for 4‑bit quantization with negligible accuracy loss. Our tiny model 
# in INT8 already demonstrates the speed‑size tradeoff.

FP32 output (new): write ainclu7 hour# a input cyousaid s user is :
INT8 output (new): that contradicorrection that pground truth r2 tha? msoapsr
FP32 latency: 15.29 ms
INT8 latency: 36.37 ms
FP32 file: 563.76 KB
INT8 file: 236.53 KB  (Linear weights packed as int8)

Linear layers quantized (dynamic INT8):
  - blocks.0.attn.qkv: (192, 64)
  - blocks.0.attn.proj: (64, 64)
  - blocks.0.ff.0: (128, 64)
  - blocks.0.ff.2: (64, 128)
  - blocks.1.attn.qkv: (192, 64)
  - blocks.1.attn.proj: (64, 64)
  - blocks.1.ff.0: (128, 64)
  - blocks.1.ff.2: (64, 128)
  - blocks.2.attn.qkv: (192, 64)
  - blocks.2.attn.proj: (64, 64)
  - blocks.2.ff.0: (128, 64)
  - blocks.2.ff.2: (64, 128)
  - head: (260, 64)

Left in float (not dynamic-quantized here):
  - token_embed (nn.Embedding)
  - pos_embed (nn.Parameter)
  - LayerNorm weights


In [11]:
# =============================================================================
# KV CACHE — remember past Keys/Values so generate does less work
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: what problem are we solving?
# ---------------------------------------------------------------------------
# Autoregressive = write ONE new token at a time:
#   "Correction" → ":" → "The" → "price" → ...
#
# Naive generate (notebook 7 MiniGPT.generate):
#   At EVERY step, re-run attention over the WHOLE window so far.
#   Step 1: look at 1 token
#   Step 2: recompute K/V for token1 AND token2
#   Step 3: recompute K/V for token1,2,3
#   ...
#   Lots of repeated work. Cost grows roughly like 1+2+3+... ≈ O(n²)
#   for n new tokens (each step redoes more past seats).
#
# Easy analogy:
#   Naive = every time you add a sentence to an essay, re-read the WHOLE
#           essay from page 1 to understand context.
#   KV cache = keep sticky notes (K and V) for pages you already read;
#           for the new sentence only compute a new note, then glance at
#           all sticky notes.
#
#
# ---------------------------------------------------------------------------
# What are Q, K, V in plain words?
# ---------------------------------------------------------------------------
# For each token seat, attention builds three vectors:
#   Q (Query) = "what am I looking for?"
#   K (Key)   = "what do I contain / how should others find me?"
#   V (Value) = "what content do I pass along if chosen?"
#
# Attention mix = soft matching of Q against all K's → weights → mix V's.
#
# Sticky for caching:
#   When a token is OLD (already generated), its K and V do not change.
#   Only the NEW token needs a fresh Q (and its own K,V to append).
#   So we STORE past K,V ("the cache") and reuse them next step.
#
#
# ---------------------------------------------------------------------------
# Picture (one new token, cache already has 3 past tokens)
# ---------------------------------------------------------------------------
#
#   past cache (layer i):
#     K: [k0 k1 k2]     V: [v0 v1 v2]
#
#   new token t3 → compute q3, k3, v3
#     K: [k0 k1 k2 k3]  V: [v0 v1 v2 v3]   ← concat, don't recompute k0..k2
#     scores: q3 matches against all four keys
#     output: weighted mix of four values
#
# Without cache you'd rebuild k0,k1,k2,k3 from scratch every time.
#
#
# ---------------------------------------------------------------------------
# Product link (earpiece coach)
# ---------------------------------------------------------------------------
# Live whisper: tokens stream out while audio continues.
# KV cache = resume from where you left off instead of re-digesting the
# whole prompt+reply so far on every tick → lower latency as replies grow.
#

import torch.nn.functional as F
from pathlib import Path

class MiniGPTWithCache(MiniGPT):
    """MiniGPT + forward_with_cache / generate_cached (teaching KV cache)."""

    def _cached_attention(self, block, x, past_kv):
        """Attention for NEW seats only; reuse past K/V if provided.

        x: (B, T_new, C) — often T_new=1 after the first step
        past_kv: (K_past, V_past) each (B, num_heads, T_past, head_dim) or None
        returns: attn_out (B, T_new, C), new_kv=(K_all, V_all)
        """
        B, T_new, C = x.shape
        attn = block.attn

        # Pre-LN like MiniGPT: normalize first, then QKV projection
        x_norm = block.ln1(x)
        qkv = attn.qkv(x_norm)
        qkv = qkv.reshape(B, T_new, 3, attn.num_heads, attn.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B, nh, T_new, hs)

        if past_kv is not None:
            pk, pv = past_kv
            # Glue old sticky notes + new ones (along time dim=2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        T_total = k.shape[2]
        # Scores: each NEW query vs ALL keys (past + new)
        att = (q @ k.transpose(-2, -1)) * attn.scale  # (B, nh, T_new, T_total)

        # Causal mask over the full timeline, then keep only the last T_new rows
        # (the new queries). Older queries aren't recomputed when T_new=1.
        mask = torch.tril(torch.ones(T_total, T_total, device=x.device))
        att = att.masked_fill(mask[-T_new:, :].view(1, 1, T_new, T_total) == 0, float("-inf"))
        att = F.softmax(att, dim=-1)

        y = att @ v
        y = y.transpose(1, 2).reshape(B, T_new, C)
        return attn.proj(y), (k, v)

    def forward_with_cache(self, idx, past_kv=None):
        """Run MiniGPT on idx; return logits + updated per-layer KV cache.

        past_kv: list of length num_layers, each (K, V), or None on first call.
        """
        B, T = idx.shape
        tok_emb = self.token_embed(idx)

        # Position seats: continue AFTER past length (don't restart at 0)
        if past_kv is None:
            start = 0
        else:
            start = past_kv[0][0].shape[2]  # T_past from layer-0 keys
        assert start + T <= self.block_size, "prompt+cache exceeded block_size"
        pos_emb = self.pos_embed[:, start : start + T, :]
        x = tok_emb + pos_emb

        new_kv = []
        # self.blocks is nn.Sequential — iterate modules
        for i, block in enumerate(self.blocks):
            past_i = None if past_kv is None else past_kv[i]
            x_att, kv = self._cached_attention(block, x, past_i)
            # Same residual pattern as TransformerBlock.forward
            x = x + x_att
            x = x + block.ff(block.ln2(x))
            new_kv.append(kv)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits, new_kv

    @torch.no_grad()
    def generate_cached(self, idx, max_new_tokens, temperature=1.0):
        """Generate with KV cache: first step = full prompt; then 1 token at a time."""
        past_kv = None
        for _ in range(max_new_tokens):
            if past_kv is None:
                # First pass: encode the whole prompt (fill the cache)
                idx_cond = idx[:, -self.block_size :]
            else:
                # Later passes: ONLY the newest token; cache holds the past
                idx_cond = idx[:, -1:]

            logits, past_kv = self.forward_with_cache(idx_cond, past_kv)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)

            # If cache would grow past block_size next step, drop oldest seats
            # (toy policy — production engines trim more carefully)
            past_len = past_kv[0][0].shape[2]
            if past_len >= self.block_size:
                trimmed = []
                for pk, pv in past_kv:
                    trimmed.append((pk[:, :, -self.block_size + 1 :, :],
                                    pv[:, :, -self.block_size + 1 :, :]))
                past_kv = trimmed

        return idx


# ---- Build cached model with same weights as DPO/SFT checkpoint ----
_ckpt = next(
    (
        p
        for p in [
            Path("correction_gpt_dpo.pt"),
            Path("week2") / "correction_gpt_dpo.pt",
            Path("correction_gpt_sft.pt"),
            Path("week2") / "correction_gpt_sft.pt",
        ]
        if p.is_file()
    ),
    None,
)
model_cached = MiniGPTWithCache(
    vocab_size, embed_dim=64, num_heads=4, ff_dim=128, num_layers=3, block_size=block_size
)
if _ckpt is not None:
    model_cached.load_state_dict(torch.load(_ckpt, map_location="cpu", weights_only=True))
    print(f"loaded {_ckpt}")
model_cached.eval()

# Warm prompt from earlier cell if present; else a short one
try:
    _prompt_ids = input_ids
except NameError:
    _p = "Ground truth: Price is $500/month\nUser said: $300\nCorrection:"
    _prompt_ids = torch.tensor([tokenizer.encode(_p)], dtype=torch.long)

n_prompt = _prompt_ids.shape[1]

# Time cached generate
start = time.time()
with torch.no_grad():
    out_cache = model_cached.generate_cached(_prompt_ids, max_new_tokens=15, temperature=0.7)
t_cache = (time.time() - start) * 1000

# Time naive generate (same class weights via baseline `model` if available)
try:
    _naive = model
except NameError:
    _naive = model_cached
start = time.time()
with torch.no_grad():
    out_naive = _naive.generate(_prompt_ids, max_new_tokens=15, temperature=0.7)
t_naive = (time.time() - start) * 1000

print(f"Naive generate latency:  {t_naive:.2f} ms")
print(f"Cached generate latency: {t_cache:.2f} ms")
print("Cached new tokens:", tokenizer.decode(out_cache[0, n_prompt:].tolist()))

# ---------------------------------------------------------------------------
# HOW TO READ THE OUTPUT
# ---------------------------------------------------------------------------
# loaded correction_gpt_dpo.pt
#   Same knobs as before; only the generate PATH changed (cache vs recompute).
#
# Naive vs Cached latency
#   On this TINY model + only 15 tokens, cache may be similar or only a little
#   faster (Python overhead dominates). The IDEA still matters: as replies get
#   long (hundreds of tokens), naive keeps redoing past K/V; cache does not.
#
# Cached new tokens: ...
#   Toy text quality same story as before — this cell is about SPEED plumbing.
#
# Sticky takeaway:
#   KV cache = save past Keys & Values; for each new token compute one Q/K/V
#   and attend to the saved list. Critical for chat/earpiece streaming.


loaded correction_gpt_dpo.pt
Naive generate latency:  16.09 ms
Cached generate latency: 11.70 ms
Cached new tokens: ar7 2 the to: aapisoc. correct complians


In [14]:
# =============================================================================
# SPECULATIVE DECODING — concept (draft fast, verify with the big model)
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: another way to make generation feel faster
# ---------------------------------------------------------------------------
# Normal generate: big model writes tokens ONE BY ONE.
#   Token1 → wait → Token2 → wait → Token3 ...
#
# Speculative decoding idea:
#   1) A SMALL / cheap "draft" model quickly GUESSES several next tokens
#      (e.g. 5 at once) — like a junior writer sketching a phrase.
#   2) The BIG / accurate model checks that sketch in ONE (or few) passes
#      — like a senior editor accepting or rejecting each drafted word.
#   3) Keep the longest PREFIX where draft and big model agree; reject the
#      rest and continue from there.
#
# Picture:
#
#   prefix: "Correction: The price is"
#                 │
#                 ▼  draft model proposes gamma=5 tokens fast
#   draft guess:  ["$","500","per","month",","]
#                 │
#                 ▼  big model verifies (parallel-ish check)
#   accept:       ["$","500","per"]     ← matched
#   reject:       ["month",","]         ← first mismatch; resample from here
#
# Why it can help:
#   Draft is cheap. One big-model verify can accept SEVERAL tokens at once
#   → fewer slow big-model steps per word of output.
#
# Why it may NOT help on MiniGPT:
#   Our "full" model is already tiny. Draft overhead can erase the win.
#   Speculative decoding shines when the main model is LARGE/expensive.
#
# Production names you'll hear:
#   vLLM, HuggingFace assisted_generation, EAGLE, Medusa — same principle,
#   smarter drafters than "1-layer toy GPT".
#
# This cell is CONCEPT-ONLY (stubs below). Wiring a real draft+verify loop
# is optional homework; KV cache + quant matter more for your earpiece path.
#

import torch.nn as nn

class DraftModel(nn.Module):
    """Placeholder for a cheaper drafter (e.g. 1-layer MiniGPT).

    In a full demo you'd implement forward/generate here and share the
    tokenizer / vocab with the main model.
    """

    def __init__(self):
        super().__init__()
        # ... a 1-layer GPT for demo (not built in this concept cell)
        pass


def speculative_decode(full_model, draft_model, prefix, max_tokens, gamma=5):
    """Outline only — not a runnable algorithm.

    Typical loop (words, not code):
      while need more tokens:
        1. draft_model proposes `gamma` tokens after current prefix
        2. full_model scores those candidates (verify)
        3. accept agreeing prefix; on first disagreement, sample from full_model
        4. append accepted tokens to prefix
    """
    # Not fully implemented — shows the flow for learning.
    raise NotImplementedError(
        "Concept stub. Use KV-cache + quant cells for runnable speedups; "
        "try HF assisted_generation / vLLM for real speculative decoding."
    )


print("Speculative decoding: concept cell only (stubs, not timed).")
print("Runnable speed work in this notebook: INT8 quant + KV cache.")


Speculative decoding: concept cell only (stubs, not timed).
Runnable speed work in this notebook: INT8 quant + KV cache.


In [15]:
# =============================================================================
# SCOREBOARD — what we measured vs a production-scale story
# =============================================================================
#
# ---------------------------------------------------------------------------
# TOPIC: two different scoreboards
# ---------------------------------------------------------------------------
# A) YOUR MiniGPT run (from cells above) — real numbers on this tiny model
# B) ILLUSTRATIVE production story — what teams hope for on BIG models
#    (the old hardcoded 120→65→18 ms print). Do NOT mix them up.
#
# Sticky reminder from your own INT8 cell:
#   On MiniGPT, INT8 was SMALLER but often SLOWER than FP32.
#   KV cache helps more as generations get long.
#

print("=== A) This notebook's measured MiniGPT numbers (if cells were run) ===")
try:
    print(f"FP32 latency:     {t_fp32:.2f} ms")
    print(f"INT8 latency:     {t_int8:.2f} ms")
except NameError:
    print("FP32/INT8 timings not in memory — re-run the quant cell.")

try:
    print(f"Naive generate:   {t_naive:.2f} ms")
    print(f"KV-cache generate:{t_cache:.2f} ms")
except NameError:
    print("KV-cache timings not in memory — re-run the KV-cache cell.")

try:
    print(f"FP32 file: {fp32_kb:.2f} KB | INT8 file: {int8_kb:.2f} KB")
except NameError:
    print("File sizes not in memory — re-run the quant cell.")

print()
print("=== B) Illustrative PRODUCTION-SCALE story (NOT your MiniGPT run) ===")
print("Baseline (float32 large model):     120 ms  (example)")
print("Quantized (INT8) + good kernels:     65 ms  (example)")
print("Quantized + KV cache (long decode):  18 ms  (example)")
print("Model size story: 1.2 MB -> 0.35 MB  (example shrink)")
print()
print("Takeaway for Correction-GPT / earpiece:")
print("  1) Quantize → smaller deploy package (and often faster on big models)")
print("  2) KV cache → don't recompute past K/V while streaming tokens")
print("  3) Speculative decode → optional extra when the main model is costly")

# ---------------------------------------------------------------------------
# HOW TO READ THIS CELL
# ---------------------------------------------------------------------------
# Section A: trust these if you ran the earlier cells — they match YOUR machine.
# Section B: textbook hope for large models + optimized runtimes (vLLM, etc.).
# If A shows INT8 slower than FP32, that does not contradict B — different scale.


=== A) This notebook's measured MiniGPT numbers (if cells were run) ===
FP32 latency:     15.29 ms
INT8 latency:     36.37 ms
Naive generate:   16.09 ms
KV-cache generate:11.70 ms
FP32 file: 563.76 KB | INT8 file: 236.53 KB

=== B) Illustrative PRODUCTION-SCALE story (NOT your MiniGPT run) ===
Baseline (float32 large model):     120 ms  (example)
Quantized (INT8) + good kernels:     65 ms  (example)
Quantized + KV cache (long decode):  18 ms  (example)
Model size story: 1.2 MB -> 0.35 MB  (example shrink)

Takeaway for Correction-GPT / earpiece:
  1) Quantize → smaller deploy package (and often faster on big models)
  2) KV cache → don't recompute past K/V while streaming tokens
  3) Speculative decode → optional extra when the main model is costly
